In [18]:
import truststore
truststore.inject_into_ssl()

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

In [ ]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

# Groq vision models (e.g. llama-4-scout) are not available on this account.
# Use Gemini for image reading instead.
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

In [8]:
@tool
def  get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "normal": {
            "eat": ["fruits", "vegetables", "whole grains", "lean proteins"],
            "avoid": ["processed foods", "excess sugar", "excess salt"]
        },
        "high_cholesterol": {
            "eat": ["oats", "barley", "beans", "nuts", "fatty fish"],
            "avoid": ["red meat", "full-fat dairy products", "fried foods"]
        },
        "high_sugar": {
            "eat": ["vegetables", "whole grains", "lean proteins"],
            "avoid": ["sweets", "sugary drinks", "processed foods"]
        }
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [ ]:
SYSTEM_PROMPT = """
You are a concise medical and nutrition assistant.
Analyze the blood work image and categorize the overall condition as exactly one of:
normal, high_cholesterol, or high_sugar.
Call get_diet_recommendation with that category.

Return only a short plain-language summary of two or three sentences:
1. State the patient's name and identified condition.
2. Briefly summarize what the patient should eat and avoid using the tool result.
Do not list individual test results, reference ranges, headings, tables, bullet points, or a disclaimer.
"""

agent = create_agent(
    llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_diet_recommendation]
)

In [17]:
from IPython.display import Markdown, display

result = agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text", "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

content = result["messages"][-1].content
if isinstance(content, list):
    response_text = "\n".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )
else:
    response_text = content

display(Markdown(response_text))

Based on the blood work report provided for **Rajesh Sharma**, here is an analysis of the results:

### **Blood Work Analysis**

1. **Complete Blood Count (CBC):** **Normal**
   * **Hemoglobin:** 15.1 g/dL (Normal: 13.5–17.5)
   * **Hematocrit:** 44% (Normal: 41%–53%)
   * **WBC:** 6.8 x10^3/uL (Normal: 4.5–11.0)
   * **Platelets:** 220 x10^3/uL (Normal: 150–400)

2. **Metabolic Panel:** **Normal**
   * **Fasting Glucose:** 92 mg/dL (Normal: 70–99)
   * **HbA1c:** 5.3% (Normal: <5.7%)
   * **Creatinine:** 1.0 mg/dL (Normal: 0.7–1.3)
   * **eGFR:** 82 mL/min (Normal: >60)

3. **Lipid Panel:** **Abnormal (High Cholesterol)**
   * **Total Cholesterol:** **238 mg/dL** (High, Normal: <200)
   * **LDL Cholesterol ("Bad" Cholesterol):** **162 mg/dL** (High, Normal: <100)
   * **HDL Cholesterol ("Good" Cholesterol):** **36 mg/dL** (Low, Normal: >40)
   * **Triglycerides:** **188 mg/dL** (High, Normal: <150)

---

### **Dietary Recommendations for High Cholesterol**

To help manage and lower your cholesterol levels, it is highly recommended to adopt a heart-healthy diet. 

#### **Foods to Include:**
* **Soluble Fiber:** Oats, barley, beans, lentils, and Brussels sprouts. Soluble fiber helps reduce the absorption of cholesterol into your bloodstream.
* **Healthy Fats:** Nuts (walnuts, almonds), seeds, and olive oil.
* **Omega-3 Fatty Acids:** Fatty fish like salmon, mackerel, and sardines.
* **Sterols and Stanols:** Foods fortified with plant sterols (like certain margarines or orange juices) can help block cholesterol absorption.

#### **Foods to Avoid or Limit:**
* **Saturated Fats:** Red meat (beef, pork, lamb) and full-fat dairy products (butter, cheese, whole milk).
* **Trans Fats:** Fried foods, commercial baked goods (cookies, cakes), and anything containing "partially hydrogenated oils."
* **Highly Processed Foods:** Fast food, chips, and sugary snacks.

*Disclaimer: Please consult with your primary care physician or a registered dietitian before making significant changes to your diet or starting any new health regimen.*